# Download and Inspect the USGS 2018 National Seismic Hazard Model

This notebook downloads a reproducible release of the USGS 2018
Conterminous United States National Seismic Hazard Model, examines its
source-model structure, and identifies the files needed to construct a
stochastic annual earthquake event catalog for the Pacific Northwest.

In [3]:
!pip install requests pandas -q



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from __future__ import annotations

from pathlib import Path
import hashlib
import zipfile

import requests


#Model details, can be changed to other models in future runs

MODEL_NAME = "nshm-conus"
MODEL_EDITION = "2018"
MODEL_TAG = "5.2.4"

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / f"usgs_nshm_conus_{MODEL_EDITION}"
)

ARCHIVE_NAME = f"{MODEL_NAME}-{MODEL_TAG}.zip"
ARCHIVE_PATH = RAW_DATA_DIR / ARCHIVE_NAME

EXTRACT_PARENT = RAW_DATA_DIR / f"{MODEL_NAME}-{MODEL_TAG}"

ARCHIVE_URL = (
    "https://code.usgs.gov/ghsc/nshmp/nshms/"
    f"{MODEL_NAME}/-/archive/{MODEL_TAG}/{ARCHIVE_NAME}"
)

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)




def calculate_sha256(file_path: Path) -> str:
    """Calculate the SHA-256 checksum of a file."""
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


def download_file(
    url: str,
    output_path: Path,
    chunk_size: int = 1024 * 1024,
) -> None:
    """Download a file while avoiding loading it entirely into memory."""

    print(f"Downloading:\n{url}")
    print(f"\nSaving to:\n{output_path}")

    headers = {
        "User-Agent": (
            "Mozilla/5.0 research-download "
            "seismic-correlation-insurance-loss"
        )
    }

    with requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=(30, 600),
        allow_redirects=True,
    ) as response:

        response.raise_for_status()

        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)

    if not zipfile.is_zipfile(output_path):
        output_path.unlink(missing_ok=True)

        raise RuntimeError(
            "The downloaded file is not a valid ZIP archive. "
            "The USGS server may have returned an HTML error page."
        )


def find_model_directory(extraction_directory: Path) -> Path:
    """Find the extracted folder containing the NSHM source directories."""

    possible_roots = [
        extraction_directory,
        *[
            path
            for path in extraction_directory.rglob("*")
            if path.is_dir()
        ],
    ]

    for candidate in possible_roots:
        if (
            (candidate / "active-crust").exists()
            and (candidate / "subduction").exists()
        ):
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate a folder containing both "
        "'active-crust' and 'subduction'."
    )



if ARCHIVE_PATH.exists() and zipfile.is_zipfile(ARCHIVE_PATH):
    print("The model archive has already been downloaded.")
else:
    download_file(
        url=ARCHIVE_URL,
        output_path=ARCHIVE_PATH,
    )

archive_size_mb = ARCHIVE_PATH.stat().st_size / (1024**2)
archive_checksum = calculate_sha256(ARCHIVE_PATH)

print("\nDownload complete.")
print(f"Archive size: {archive_size_mb:,.2f} MB")
print(f"SHA-256: {archive_checksum}")




if not EXTRACT_PARENT.exists():
    EXTRACT_PARENT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ARCHIVE_PATH, "r") as zip_file:
        zip_file.extractall(EXTRACT_PARENT)

    print("\nArchive extracted.")
else:
    print("\nExtraction directory already exists.")




MODEL_DIR = find_model_directory(EXTRACT_PARENT)

print("\nUSGS model directory:")
print(MODEL_DIR)

print("\nTop-level model contents:")

for path in sorted(MODEL_DIR.iterdir()):
    object_type = "DIR " if path.is_dir() else "FILE"
    print(f"{object_type}: {path.name}")

Downloading:
https://code.usgs.gov/ghsc/nshmp/nshms/nshm-conus/-/archive/5.2.4/nshm-conus-5.2.4.zip

Saving to:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\raw\usgs_nshm_conus_2018\nshm-conus-5.2.4.zip

Download complete.
Archive size: 19.24 MB
SHA-256: c1b6e73f303ee4cee1ad057714d073eb9528c17168fee779c6c458df19dc0b47

Archive extracted.

USGS model directory:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\raw\usgs_nshm_conus_2018\nshm-conus-5.2.4\nshm-conus-5.2.4

Top-level model contents:
DIR : active-crust
FILE: calc-config.json
FILE: code.json
FILE: CODE_OF_CONDUCT.md
FILE: CONTRIBUTING.md
FILE: DISCLAIMER.md
FILE: LICENSE.md
FILE: map.geojson
FILE: model-info.json
FILE: README.md
DIR : site-data
FILE: sites.csv
FILE: sites.geojson
DIR : stable-crust
DIR : subduction


In [5]:
#This protects raw and large files from my githib page
from pathlib import Path


gitignore_path = PROJECT_ROOT / ".gitignore"

gitignore_entries = [
    "# Raw USGS seismic hazard model",
    "data/raw/usgs_nshm_conus_2018/",
    "",
    "# Generated event catalogs and large calculation files",
    "data/processed/event_catalogs/",
    "*.hdf5",
    "*.h5",
    "*.parquet",
    "",
    "# Python and Jupyter temporary files",
    "__pycache__/",
    "*.py[cod]",
    ".ipynb_checkpoints/",
]

existing_lines = set()

if gitignore_path.exists():
    existing_lines = set(
        gitignore_path.read_text(encoding="utf-8").splitlines()
    )

lines_to_add = [
    line
    for line in gitignore_entries
    if line not in existing_lines
]

if lines_to_add:
    with gitignore_path.open("a", encoding="utf-8") as file:
        if gitignore_path.stat().st_size > 0:
            file.write("\n")

        file.write("\n".join(lines_to_add))
        file.write("\n")

    print("New entries added to .gitignore.")
else:
    print("All required entries are already in .gitignore.")

print(f"\n.gitignore location:\n{gitignore_path}")

print("\nCurrent .gitignore contents:\n")
print(gitignore_path.read_text(encoding="utf-8"))

New entries added to .gitignore.

.gitignore location:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\.gitignore

Current .gitignore contents:

# Raw USGS seismic hazard model
data/raw/usgs_nshm_conus_2018/

# Generated event catalogs and large calculation files
data/processed/event_catalogs/
*.hdf5
*.h5
*.parquet

# Python and Jupyter temporary files
__pycache__/
*.py[cod]
.ipynb_checkpoints/



In [6]:
# This is a repository of the usgs model files 

from pathlib import Path

import pandas as pd
from IPython.display import display


file_records: list[dict[str, object]] = []

for file_path in MODEL_DIR.rglob("*"):
    if not file_path.is_file():
        continue

    relative_path = file_path.relative_to(MODEL_DIR)

    file_records.append(
        {
            "relative_path": relative_path.as_posix(),
            "top_level_group": relative_path.parts[0],
            "file_name": file_path.name,
            "extension": (
                file_path.suffix.lower()
                if file_path.suffix
                else "[no extension]"
            ),
            "size_kb": file_path.stat().st_size / 1024,
        }
    )


manifest = (
    pd.DataFrame(file_records)
    .sort_values("relative_path")
    .reset_index(drop=True)
)


# Save a lightweight inventory that can be committed to GitHub.
metadata_directory = PROJECT_ROOT / "data" / "metadata"
metadata_directory.mkdir(parents=True, exist_ok=True)

manifest_path = (
    metadata_directory
    / "usgs_nshm_2018_file_manifest.csv"
)

manifest.to_csv(manifest_path, index=False)


# Summarize the major source-model categories.
group_summary = (
    manifest.groupby("top_level_group", as_index=False)
    .agg(
        number_of_files=("relative_path", "count"),
        total_size_mb=(
            "size_kb",
            lambda values: values.sum() / 1024,
        ),
    )
    .sort_values(
        "number_of_files",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(f"Total files inventoried: {len(manifest):,}")
print(f"\nManifest saved to:\n{manifest_path}")

print("\nFiles by major source-model group:")
display(group_summary)

print("\nFirst 20 files in the model:")
display(manifest.head(20))

Total files inventoried: 895

Manifest saved to:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\usgs_nshm_2018_file_manifest.csv

Files by major source-model group:


,top_level_group,number_of_files,total_size_mb
0,active-crust,604,72.205391
1,stable-crust,183,9.185053
2,subduction,89,0.231301
3,site-data,8,1.316826
4,CODE_OF_CONDUCT.md,1,0.000249
5,CONTRIBUTING.md,1,0.000665
6,DISCLAIMER.md,1,0.000594
7,LICENSE.md,1,0.001544
8,README.md,1,0.001902
9,calc-config.json,1,0.000086



First 20 files in the model:


,relative_path,top_level_group,file_name,extension,size_kb
0,CODE_OF_CONDUCT.md,CODE_OF_CONDUCT.md,CODE_OF_CONDUCT.md,.md,0.254883
1,CONTRIBUTING.md,CONTRIBUTING.md,CONTRIBUTING.md,.md,0.680664
2,DISCLAIMER.md,DISCLAIMER.md,DISCLAIMER.md,.md,0.608398
3,LICENSE.md,LICENSE.md,LICENSE.md,.md,1.581055
4,README.md,README.md,README.md,.md,1.947266
5,active-crust/fault/AZ/Algodones.geojson,active-crust,Algodones.geojson,.geojson,0.972656
6,active-crust/fault/AZ/Aubrey.geojson,active-crust,Aubrey.geojson,.geojson,1.203125
7,active-crust/fault/AZ/Big Chino - Little Chino...,active-crust,Big Chino - Little Chino.geojson,.geojson,1.797852
8,active-crust/fault/AZ/Dutchman Draw.geojson,active-crust,Dutchman Draw.geojson,.geojson,0.836914
9,active-crust/fault/AZ/Hurricane (center).geojson,active-crust,Hurricane (center).geojson,.geojson,1.607422


In [ ]:
#This is to classify the cascadia subduction sources that are need for the project

import pandas as pd
from IPython.display import display


# Select only files contained in the subduction model.
subduction_manifest = manifest.loc[
    manifest["relative_path"].str.startswith("subduction/")
].copy()


def classify_subduction_file(relative_path: str) -> str:
    """Assign each subduction file to a meaningful model category."""

    if relative_path.startswith(
        "subduction/interface/Cascadia/"
    ):
        return "Cascadia interface"

    if relative_path.startswith(
        "subduction/slab/OR/"
    ):
        return "Oregon intraslab"

    if relative_path.startswith(
        "subduction/slab/WA/"
    ):
        return "Washington intraslab"

    if relative_path.startswith(
        "subduction/slab/CA/"
    ):
        return "California intraslab"

    if relative_path.startswith(
        "subduction/interface/"
    ):
        return "General interface configuration"

    if relative_path.startswith(
        "subduction/slab/"
    ):
        return "General slab configuration"

    return "Other subduction file"


subduction_manifest["subduction_category"] = (
    subduction_manifest["relative_path"].apply(
        classify_subduction_file
    )
)


# Summarize the number and size of files in each category.
subduction_summary = (
    subduction_manifest.groupby(
        "subduction_category",
        as_index=False,
    )
    .agg(
        number_of_files=("relative_path", "count"),
        total_size_kb=("size_kb", "sum"),
    )
    .sort_values(
        "number_of_files",
        ascending=False,
    )
    .reset_index(drop=True)
)


# Save the subduction-only manifest.
subduction_manifest_path = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "usgs_nshm_2018_subduction_manifest.csv"
)

subduction_manifest.to_csv(
    subduction_manifest_path,
    index=False,
)


print(
    f"Total subduction files: "
    f"{len(subduction_manifest):,}"
)

print(
    "\nSubduction manifest saved to:\n"
    f"{subduction_manifest_path}"
)

print("\nSubduction model summary:")
display(subduction_summary)


print("\nCascadia interface files:")

cascadia_files = (
    subduction_manifest.loc[
        subduction_manifest[
            "subduction_category"
        ].eq("Cascadia interface"),
        [
            "relative_path",
            "extension",
            "size_kb",
        ],
    ]
    .sort_values("relative_path")
    .reset_index(drop=True)
)

display(cascadia_files)


print("\nOregon intraslab files:")

oregon_slab_files = (
    subduction_manifest.loc[
        subduction_manifest[
            "subduction_category"
        ].eq("Oregon intraslab"),
        [
            "relative_path",
            "extension",
            "size_kb",
        ],
    ]
    .sort_values("relative_path")
    .reset_index(drop=True)
)

display(oregon_slab_files)

Total subduction files: 89

Subduction manifest saved to:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\usgs_nshm_2018_subduction_manifest.csv

Subduction model summary:


,subduction_category,number_of_files,total_size_kb
0,Cascadia interface,56,21.590820
1,General slab configuration,12,213.159180
2,Oregon intraslab,6,0.549805
3,Washington intraslab,6,0.550781
4,General interface configuration,5,0.697266
5,California intraslab,4,0.304688



Cascadia interface files:


,relative_path,extension,size_kb
0,subduction/interface/Cascadia/README.md,.md,3.117188
1,subduction/interface/Cascadia/bottom/full-rupt...,.json,0.120117
2,subduction/interface/Cascadia/bottom/partial-r...,.json,0.095703
3,subduction/interface/Cascadia/bottom/partial-r...,.json,0.095703
4,subduction/interface/Cascadia/bottom/partial-r...,.json,0.136719
5,subduction/interface/Cascadia/bottom/partial-r...,.json,0.126953
6,subduction/interface/Cascadia/bottom/partial-r...,.json,0.203125
7,subduction/interface/Cascadia/bottom/partial-r...,.json,0.102539
8,subduction/interface/Cascadia/bottom/partial-r...,.json,0.135742
9,subduction/interface/Cascadia/bottom/partial-r...,.json,0.049805



Oregon intraslab files:


,relative_path,extension,size_kb
0,subduction/slab/OR/mfd-hi/rate-tree.json,.json,0.078125
1,subduction/slab/OR/mfd-hi/rupture-sets.json,.json,0.142578
2,subduction/slab/OR/mfd-lo/rate-tree.json,.json,0.078125
3,subduction/slab/OR/mfd-lo/rupture-sets.json,.json,0.139648
4,subduction/slab/OR/source-group.json,.json,0.057617
5,subduction/slab/OR/tree-info.json,.json,0.053711


In [ ]:
# Here, we are reading the cascadia and oregon subduction model definitions 


from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def load_model_json(relative_path: str):
    """Load a JSON file relative to the USGS model directory."""

    file_path = MODEL_DIR / Path(relative_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"USGS model file was not found:\n{file_path}"
        )

    with file_path.open("r", encoding="utf-8") as file:
        return json.load(file)


# Cascadia geometry logic tree

cascadia_geometry_data = load_model_json(
    "subduction/interface/Cascadia/source-tree.json"
)

cascadia_geometry_tree = pd.DataFrame(
    cascadia_geometry_data
).rename(
    columns={
        "id": "geometry_branch",
        "weight": "branch_weight",
    }
)

cascadia_geometry_tree["weight_sum_check"] = (
    cascadia_geometry_tree["branch_weight"].sum()
)

print("Cascadia geometry logic tree:")
display(cascadia_geometry_tree)

print(
    "Sum of Cascadia geometry weights:",
    cascadia_geometry_tree["branch_weight"].sum(),
)

#magnitude and frequency distributions for cascadia


cascadia_mfd_map = load_model_json(
    "subduction/interface/Cascadia/mfd-map.json"
)

cascadia_mfd_records: list[dict[str, object]] = []

for mfd_tree_name, branches in cascadia_mfd_map.items():
    for branch in branches:

        value = branch.get("value", {})

        cascadia_mfd_records.append(
            {
                "mfd_tree_name": mfd_tree_name,
                "branch_id": branch.get("id"),
                "branch_weight": branch.get("weight"),
                "mfd_type": value.get("type"),
                "magnitude": value.get("m"),
                "annual_rate": value.get("rate"),
                "a_parameter": value.get("a"),
                "b_parameter": value.get("b"),
                "minimum_magnitude": value.get("mMin"),
                "maximum_magnitude": value.get("mMax"),
                "magnitude_increment": value.get("Δm"),
            }
        )

cascadia_mfd_branches = (
    pd.DataFrame(cascadia_mfd_records)
    .sort_values(
        [
            "mfd_tree_name",
            "branch_id",
        ]
    )
    .reset_index(drop=True)
)

print("\nCascadia magnitude-frequency branches:")
display(cascadia_mfd_branches)


#partial rupture logic tree

partial_tree_records: list[dict[str, object]] = []

for geometry in ["bottom", "middle", "top"]:

    segmented_tree = load_model_json(
        "subduction/interface/Cascadia/"
        f"{geometry}/partial-rupture/source-tree.json"
    )

    for branch in segmented_tree:
        partial_tree_records.append(
            {
                "geometry_branch": geometry,
                "logic_tree_level": (
                    "segmented versus unsegmented"
                ),
                "branch_id": branch.get("id"),
                "branch_weight": branch.get("weight"),
            }
        )

    unsegmented_tree = load_model_json(
        "subduction/interface/Cascadia/"
        f"{geometry}/partial-rupture/"
        "unsegmented/source-tree.json"
    )

    for branch in unsegmented_tree:
        partial_tree_records.append(
            {
                "geometry_branch": geometry,
                "logic_tree_level": (
                    "unsegmented rupture extent"
                ),
                "branch_id": branch.get("id"),
                "branch_weight": branch.get("weight"),
            }
        )

cascadia_partial_tree = (
    pd.DataFrame(partial_tree_records)
    .sort_values(
        [
            "geometry_branch",
            "logic_tree_level",
            "branch_id",
        ]
    )
    .reset_index(drop=True)
)

print("\nCascadia partial-rupture logic trees:")
display(cascadia_partial_tree)

#intraslab source group


oregon_source_group_data = load_model_json(
    "subduction/slab/OR/source-group.json"
)

oregon_source_groups = pd.DataFrame(
    oregon_source_group_data
).rename(
    columns={
        "id": "source_group_id",
    }
)

print("\nOregon intraslab source groups:")
display(oregon_source_groups)

#intraslab mfd and rate tree information

slab_mfd_map = load_model_json(
    "subduction/slab/mfd-map.json"
)

oregon_rate_records: list[dict[str, object]] = []

for source_group_id in ["mfd-hi", "mfd-lo"]:

    rate_tree = load_model_json(
        "subduction/slab/OR/"
        f"{source_group_id}/rate-tree.json"
    )

    rupture_sets = load_model_json(
        "subduction/slab/OR/"
        f"{source_group_id}/rupture-sets.json"
    )

    if len(rupture_sets) != 1:
        raise ValueError(
            f"Expected one Oregon rupture set for "
            f"{source_group_id}, but found "
            f"{len(rupture_sets)}."
        )

    rupture_set = rupture_sets[0]
    mfd_tree_name = rupture_set["mfd-tree"]
    mfd_branches = slab_mfd_map[mfd_tree_name]

    for rate_branch in rate_tree:
        for mfd_branch in mfd_branches:

            mfd_value = mfd_branch.get("value", {})

            oregon_rate_records.append(
                {
                    "source_group_id": source_group_id,
                    "rupture_set_name": rupture_set.get("name"),
                    "rupture_set_id": rupture_set.get("id"),
                    "feature_id": rupture_set.get("feature"),
                    "spatial_pdf": rupture_set.get("spatial-pdf"),
                    "mfd_tree_name": mfd_tree_name,
                    "rate_branch_id": rate_branch.get("id"),
                    "rate_branch_weight": rate_branch.get(
                        "weight"
                    ),
                    "rate_tree_value": rate_branch.get("value"),
                    "mfd_branch_id": mfd_branch.get("id"),
                    "mfd_branch_weight": mfd_branch.get(
                        "weight"
                    ),
                    "mfd_type": mfd_value.get("type"),
                    "b_parameter": mfd_value.get("b"),
                    "minimum_magnitude": mfd_value.get("mMin"),
                    "maximum_magnitude": mfd_value.get("mMax"),
                    "magnitude_increment": mfd_value.get("Δm"),
                }
            )

oregon_intraslab_branches = (
    pd.DataFrame(oregon_rate_records)
    .sort_values(
        [
            "source_group_id",
            "rate_branch_id",
            "mfd_branch_id",
        ]
    )
    .reset_index(drop=True)
)

print("\nOregon intraslab model branches:")
display(oregon_intraslab_branches)


#save your metadata

metadata_directory = PROJECT_ROOT / "data" / "metadata"
metadata_directory.mkdir(
    parents=True,
    exist_ok=True,
)

cascadia_geometry_tree.to_csv(
    metadata_directory
    / "cascadia_geometry_logic_tree.csv",
    index=False,
)

cascadia_mfd_branches.to_csv(
    metadata_directory
    / "cascadia_mfd_branches.csv",
    index=False,
)

cascadia_partial_tree.to_csv(
    metadata_directory
    / "cascadia_partial_rupture_logic_tree.csv",
    index=False,
)

oregon_intraslab_branches.to_csv(
    metadata_directory
    / "oregon_intraslab_model_branches.csv",
    index=False,
)

print("\nExtracted model metadata saved in:")
print(metadata_directory)

Cascadia geometry logic tree:


,geometry_branch,branch_weight,weight_sum_check
0,bottom,0.3,1.0
1,middle,0.5,1.0
2,top,0.2,1.0


Sum of Cascadia geometry weights: 1.0

Cascadia magnitude-frequency branches:


,mfd_tree_name,branch_id,branch_weight,mfd_type,magnitude,annual_rate,a_parameter,b_parameter,minimum_magnitude,maximum_magnitude,magnitude_increment
0,full-bottom,M1,0.334,SINGLE,8.85,0.001900,NaN,NaN,NaN,NaN,NaN
1,full-bottom,M2,0.333,SINGLE,9.01,0.001900,NaN,NaN,NaN,NaN,NaN
2,full-bottom,M3,0.333,SINGLE,9.34,0.001900,NaN,NaN,NaN,NaN,NaN
3,full-middle,M1,0.334,SINGLE,8.69,0.001900,NaN,NaN,NaN,NaN,NaN
4,full-middle,M2,0.333,SINGLE,8.82,0.001900,NaN,NaN,NaN,NaN,NaN
5,full-middle,M3,0.333,SINGLE,9.12,0.001900,NaN,NaN,NaN,NaN,NaN
6,full-top,M1,0.334,SINGLE,8.61,0.001900,NaN,NaN,NaN,NaN,NaN
7,full-top,M2,0.333,SINGLE,8.72,0.001900,NaN,NaN,NaN,NaN,NaN
8,full-top,M3,0.333,SINGLE,9.01,0.001900,NaN,NaN,NaN,NaN,NaN
9,segmented-1-2-3-bottom,M1,0.334,SINGLE,8.65,0.000174,NaN,NaN,NaN,NaN,NaN



Cascadia partial-rupture logic trees:


...


Oregon intraslab source groups:


,source_group_id
0,mfd-hi
1,mfd-lo



Oregon intraslab model branches:


,source_group_id,rupture_set_name,rupture_set_id,feature_id,spatial_pdf,mfd_tree_name,rate_branch_id,rate_branch_weight,rate_tree_value,mfd_branch_id,mfd_branch_weight,mfd_type,b_parameter,minimum_magnitude,maximum_magnitude,magnitude_increment
0,mfd-hi,OR Intraslab,8212,8210,pdf-or.csv,or-wa-slab-mfd-hi,R1,1.0,10.801016,M1,0.9,GR,0.8,7.2,7.5,0.1
1,mfd-hi,OR Intraslab,8212,8210,pdf-or.csv,or-wa-slab-mfd-hi,R1,1.0,10.801016,M2,0.1,GR,0.8,7.2,8.0,0.1
2,mfd-lo,OR Intraslab,8211,8210,pdf-or.csv,or-slab-mfd-lo,R1,1.0,0.144321,M1,1.0,GR,0.4,6.5,7.2,0.1



Extracted model metadata saved in:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata


In [9]:

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def load_json_file(relative_path: str | Path):
    """Load a JSON file relative to the USGS model directory."""

    file_path = MODEL_DIR / Path(relative_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required USGS model file was not found:\n{file_path}"
        )

    with file_path.open("r", encoding="utf-8") as file:
        return json.load(file)

#read the cascadia geometry weights


geometry_tree = load_json_file(
    "subduction/interface/Cascadia/source-tree.json"
)

geometry_weights = {
    branch["id"]: float(branch["weight"])
    for branch in geometry_tree
}

if not abs(sum(geometry_weights.values()) - 1.0) < 1e-10:
    raise ValueError(
        "Cascadia geometry weights do not sum to 1."
    )

#we need the cascadia rupture set


rupture_set_records: list[dict[str, object]] = []

cascadia_root = Path(
    "subduction/interface/Cascadia"
)


for geometry_branch, geometry_weight in geometry_weights.items():

    geometry_directory = (
        cascadia_root / geometry_branch
    )

    # ------------------------------------------------------------------------
    # A. FULL-MARGIN RUPTURE
    #
    # The full-margin model and partial-rupture model are additive.
    # There is therefore no probability split between full and partial.
    # ------------------------------------------------------------------------

    full_rupture_path = (
        geometry_directory
        / "full-rupture"
        / "rupture-set.json"
    )

    full_rupture = load_json_file(
        full_rupture_path
    )

    full_sections = full_rupture.get(
        "sections",
        [full_rupture["id"]],
    )

    rupture_set_records.append(
        {
            "geometry_branch": geometry_branch,
            "geometry_weight": geometry_weight,
            "rupture_family": "full",
            "partial_model_branch": "not_applicable",
            "partial_model_weight": 1.0,
            "rupture_extent": "sections-1-2-3-4",
            "extent_weight": 1.0,
            "source_scale": 1.0,
            "rupture_set_name": full_rupture["name"],
            "rupture_set_id": full_rupture["id"],
            "sections": "|".join(
                str(section)
                for section in full_sections
            ),
            "number_of_sections": len(full_sections),
            "mfd_tree_name": full_rupture["mfd-tree"],
            "rupture_set_path": (
                full_rupture_path.as_posix()
            ),
        }
    )

    # ------------------------------------------------------------------------
    # B. PARTIAL-RUPTURE BRANCH
    #
    # The partial model has a 50/50 split between:
    #   1. segmented
    #   2. unsegmented
    # ------------------------------------------------------------------------

    partial_tree_path = (
        geometry_directory
        / "partial-rupture"
        / "source-tree.json"
    )

    partial_tree = load_json_file(
        partial_tree_path
    )

    partial_weights = {
        branch["id"]: float(branch["weight"])
        for branch in partial_tree
    }

    if not abs(sum(partial_weights.values()) - 1.0) < 1e-10:
        raise ValueError(
            f"Partial-rupture weights do not sum to 1 "
            f"for geometry '{geometry_branch}'."
        )

    # ------------------------------------------------------------------------
    # B1. SEGMENTED PARTIAL RUPTURES
    #
    # The individual segmented rupture sets are additive within the
    # segmented branch. Their scale factors are not probabilities.
    # ------------------------------------------------------------------------

    segmented_directory = (
        geometry_directory
        / "partial-rupture"
        / "segmented"
    )

    segmented_source_group = load_json_file(
        segmented_directory
        / "source-group.json"
    )

    segmented_scales = {
        source["id"]: float(
            source.get("scale", 1.0)
        )
        for source in segmented_source_group
    }

    for rupture_extent, source_scale in segmented_scales.items():

        rupture_path = (
            segmented_directory
            / rupture_extent
            / "rupture-set.json"
        )

        rupture = load_json_file(
            rupture_path
        )

        sections = rupture.get(
            "sections",
            [rupture["id"]],
        )

        rupture_set_records.append(
            {
                "geometry_branch": geometry_branch,
                "geometry_weight": geometry_weight,
                "rupture_family": "partial",
                "partial_model_branch": "segmented",
                "partial_model_weight": (
                    partial_weights["segmented"]
                ),
                "rupture_extent": rupture_extent,
                "extent_weight": 1.0,
                "source_scale": source_scale,
                "rupture_set_name": rupture["name"],
                "rupture_set_id": rupture["id"],
                "sections": "|".join(
                    str(section)
                    for section in sections
                ),
                "number_of_sections": len(sections),
                "mfd_tree_name": rupture["mfd-tree"],
                "rupture_set_path": (
                    rupture_path.as_posix()
                ),
            }
        )

    # ------------------------------------------------------------------------
    # B2. UNSEGMENTED PARTIAL RUPTURES
    #
    # The two rupture extents are alternatives:
    #   sections 1-2-3:   weight 0.75
    #   sections 1-2-3-4: weight 0.25
    # ------------------------------------------------------------------------

    unsegmented_directory = (
        geometry_directory
        / "partial-rupture"
        / "unsegmented"
    )

    unsegmented_tree = load_json_file(
        unsegmented_directory
        / "source-tree.json"
    )

    unsegmented_extent_weights = {
        branch["id"]: float(branch["weight"])
        for branch in unsegmented_tree
    }

    if not abs(
        sum(unsegmented_extent_weights.values()) - 1.0
    ) < 1e-10:
        raise ValueError(
            f"Unsegmented extent weights do not sum to 1 "
            f"for geometry '{geometry_branch}'."
        )

    for (
        rupture_extent,
        extent_weight,
    ) in unsegmented_extent_weights.items():

        extent_directory = (
            unsegmented_directory
            / rupture_extent
        )

        extent_source_group = load_json_file(
            extent_directory
            / "source-group.json"
        )

        if len(extent_source_group) != 1:
            raise ValueError(
                "Expected one child source under "
                f"{extent_directory}, but found "
                f"{len(extent_source_group)}."
            )

        child_source = extent_source_group[0]

        child_id = child_source["id"]
        source_scale = float(
            child_source.get("scale", 1.0)
        )

        rupture_path = (
            extent_directory
            / child_id
            / "rupture-set.json"
        )

        rupture = load_json_file(
            rupture_path
        )

        sections = rupture.get(
            "sections",
            [rupture["id"]],
        )

        rupture_set_records.append(
            {
                "geometry_branch": geometry_branch,
                "geometry_weight": geometry_weight,
                "rupture_family": "partial",
                "partial_model_branch": "unsegmented",
                "partial_model_weight": (
                    partial_weights["unsegmented"]
                ),
                "rupture_extent": rupture_extent,
                "extent_weight": extent_weight,
                "source_scale": source_scale,
                "rupture_set_name": rupture["name"],
                "rupture_set_id": rupture["id"],
                "sections": "|".join(
                    str(section)
                    for section in sections
                ),
                "number_of_sections": len(sections),
                "mfd_tree_name": rupture["mfd-tree"],
                "rupture_set_path": (
                    rupture_path.as_posix()
                ),
            }
        )

#create a rupture-set table


cascadia_rupture_sets = (
    pd.DataFrame(rupture_set_records)
    .sort_values(
        [
            "geometry_branch",
            "rupture_family",
            "partial_model_branch",
            "rupture_extent",
        ]
    )
    .reset_index(drop=True)
)


# Logic-tree weight before applying the MFD branch weight.
#
# Source scale is deliberately kept separate because it modifies
# occurrence rates and is not an epistemic probability.
cascadia_rupture_sets[
    "epistemic_weight_before_mfd"
] = (
    cascadia_rupture_sets["geometry_weight"]
    * cascadia_rupture_sets["partial_model_weight"]
    * cascadia_rupture_sets["extent_weight"]
)

#validate the expected rupture set counts


expected_counts = {
    "full": 3,
    "segmented": 12,
    "unsegmented": 6,
}

actual_full = int(
    (
        cascadia_rupture_sets["rupture_family"]
        == "full"
    ).sum()
)

actual_segmented = int(
    (
        cascadia_rupture_sets["partial_model_branch"]
        == "segmented"
    ).sum()
)

actual_unsegmented = int(
    (
        cascadia_rupture_sets["partial_model_branch"]
        == "unsegmented"
    ).sum()
)

actual_counts = {
    "full": actual_full,
    "segmented": actual_segmented,
    "unsegmented": actual_unsegmented,
}

if actual_counts != expected_counts:
    raise ValueError(
        "Unexpected Cascadia rupture-set counts.\n"
        f"Expected: {expected_counts}\n"
        f"Found: {actual_counts}"
    )

#save the cascadia rupture set inventory


metadata_directory = (
    PROJECT_ROOT / "data" / "metadata"
)

metadata_directory.mkdir(
    parents=True,
    exist_ok=True,
)

rupture_inventory_path = (
    metadata_directory
    / "cascadia_rupture_set_inventory.csv"
)

cascadia_rupture_sets.to_csv(
    rupture_inventory_path,
    index=False,
)

#inspect the oregon intraslab spatial pdf


oregon_pdf_path = (
    MODEL_DIR
    / "subduction"
    / "slab"
    / "grid-data"
    / "pdf-or.csv"
)

oregon_spatial_pdf = pd.read_csv(
    oregon_pdf_path
)

required_pdf_columns = {
    "lon",
    "lat",
    "depth",
    "pdf",
}

missing_pdf_columns = (
    required_pdf_columns
    .difference(oregon_spatial_pdf.columns)
)

if missing_pdf_columns:
    raise ValueError(
        "Oregon spatial PDF is missing columns: "
        f"{sorted(missing_pdf_columns)}"
    )

pdf_sum = float(
    oregon_spatial_pdf["pdf"].sum()
)

if not abs(pdf_sum - 1.0) < 1e-8:
    raise ValueError(
        "Oregon intraslab spatial PDF does not sum "
        f"to 1. Found {pdf_sum:.12f}."
    )

oregon_pdf_summary = pd.DataFrame(
    [
        {
            "number_of_grid_points": len(
                oregon_spatial_pdf
            ),
            "pdf_sum": pdf_sum,
            "minimum_longitude": (
                oregon_spatial_pdf["lon"].min()
            ),
            "maximum_longitude": (
                oregon_spatial_pdf["lon"].max()
            ),
            "minimum_latitude": (
                oregon_spatial_pdf["lat"].min()
            ),
            "maximum_latitude": (
                oregon_spatial_pdf["lat"].max()
            ),
            "minimum_depth_km": (
                oregon_spatial_pdf["depth"].min()
            ),
            "maximum_depth_km": (
                oregon_spatial_pdf["depth"].max()
            ),
            "number_of_depth_levels": (
                oregon_spatial_pdf["depth"].nunique()
            ),
        }
    ]
)

oregon_pdf_summary_path = (
    metadata_directory
    / "oregon_intraslab_spatial_pdf_summary.csv"
)

oregon_pdf_summary.to_csv(
    oregon_pdf_summary_path,
    index=False,
)

#results

print(
    "Cascadia rupture sets extracted:",
    len(cascadia_rupture_sets),
)

print(
    "\nRupture-set inventory saved to:\n"
    f"{rupture_inventory_path}"
)

display(
    cascadia_rupture_sets[
        [
            "geometry_branch",
            "rupture_family",
            "partial_model_branch",
            "rupture_extent",
            "source_scale",
            "rupture_set_id",
            "sections",
            "mfd_tree_name",
            "epistemic_weight_before_mfd",
        ]
    ]
)

print("\nOregon intraslab spatial PDF summary:")
display(oregon_pdf_summary)

print(
    "\nOregon PDF summary saved to:\n"
    f"{oregon_pdf_summary_path}"
)

Cascadia rupture sets extracted: 21

Rupture-set inventory saved to:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\cascadia_rupture_set_inventory.csv


,geometry_branch,rupture_family,partial_model_branch,rupture_extent,source_scale,rupture_set_id,sections,mfd_tree_name,epistemic_weight_before_mfd
0,bottom,full,not_applicable,sections-1-2-3-4,1.0000,3170,3110|3120|3130|3140,full-bottom,0.3000
1,bottom,partial,segmented,section-1,1.2000,3110,3110,segmented-1-bottom,0.1500
2,bottom,partial,segmented,section-4,0.2500,3140,3140,segmented-4-bottom,0.1500
3,bottom,partial,segmented,sections-1-2,1.2000,3150,3110|3120,segmented-1-2-bottom,0.1500
4,bottom,partial,segmented,sections-1-2-3,1.2000,3160,3110|3120|3130,segmented-1-2-3-bottom,0.1500
5,bottom,partial,unsegmented,sections-1-2-3,1.2000,3160,3110|3120|3130,unsegmented,0.1125
6,bottom,partial,unsegmented,sections-1-2-3-4,1.8560,3170,3110|3120|3130|3140,unsegmented,0.0375
7,middle,full,not_applicable,sections-1-2-3-4,1.0000,3171,3111|3121|3131|3141,full-middle,0.5000
8,middle,partial,segmented,section-1,1.2000,3111,3111,segmented-1-middle,0.2500
9,middle,partial,segmented,section-4,0.2500,3141,3141,segmented-4-middle,0.2500



Oregon intraslab spatial PDF summary:


,number_of_grid_points,pdf_sum,minimum_longitude,maximum_longitude,minimum_latitude,maximum_latitude,minimum_depth_km,maximum_depth_km,number_of_depth_levels
0,821,1.0,-124.3,-122.3,42.1,46.3,42.0,60.0,3



Oregon PDF summary saved to:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\oregon_intraslab_spatial_pdf_summary.csv


In [10]:


from pathlib import Path

import pandas as pd
from IPython.display import display


#validations for output files 

required_metadata_files = {
    "Complete USGS file manifest":
        "usgs_nshm_2018_file_manifest.csv",

    "Subduction file manifest":
        "usgs_nshm_2018_subduction_manifest.csv",

    "Cascadia geometry logic tree":
        "cascadia_geometry_logic_tree.csv",

    "Cascadia MFD branches":
        "cascadia_mfd_branches.csv",

    "Cascadia partial-rupture logic tree":
        "cascadia_partial_rupture_logic_tree.csv",

    "Oregon intraslab branches":
        "oregon_intraslab_model_branches.csv",

    "Cascadia rupture-set inventory":
        "cascadia_rupture_set_inventory.csv",

    "Oregon spatial PDF summary":
        "oregon_intraslab_spatial_pdf_summary.csv",
}


validation_records = []

for description, file_name in required_metadata_files.items():

    file_path = metadata_directory / file_name

    validation_records.append(
        {
            "description": description,
            "file_name": file_name,
            "exists": file_path.exists(),
            "size_kb": (
                file_path.stat().st_size / 1024
                if file_path.exists()
                else None
            ),
        }
    )


validation_table = pd.DataFrame(validation_records)

display(validation_table)


if not validation_table["exists"].all():
    missing_files = validation_table.loc[
        ~validation_table["exists"],
        "file_name",
    ].tolist()

    raise FileNotFoundError(
        "The following required files are missing:\n"
        + "\n".join(missing_files)
    )

#validate rupture counts 


rupture_count_summary = pd.DataFrame(
    [
        {
            "rupture_category": "Full-margin",
            "expected_count": 3,
            "actual_count": int(
                (
                    cascadia_rupture_sets[
                        "rupture_family"
                    ] == "full"
                ).sum()
            ),
        },
        {
            "rupture_category": "Segmented partial",
            "expected_count": 12,
            "actual_count": int(
                (
                    cascadia_rupture_sets[
                        "partial_model_branch"
                    ] == "segmented"
                ).sum()
            ),
        },
        {
            "rupture_category": "Unsegmented partial",
            "expected_count": 6,
            "actual_count": int(
                (
                    cascadia_rupture_sets[
                        "partial_model_branch"
                    ] == "unsegmented"
                ).sum()
            ),
        },
    ]
)

rupture_count_summary["passes"] = (
    rupture_count_summary["expected_count"]
    == rupture_count_summary["actual_count"]
)

print("Cascadia rupture-set count validation:")
display(rupture_count_summary)


if not rupture_count_summary["passes"].all():
    raise ValueError(
        "One or more Cascadia rupture-set counts are incorrect."
    )

#validate geometry branches


geometry_validation = (
    cascadia_rupture_sets[
        [
            "geometry_branch",
            "geometry_weight",
        ]
    ]
    .drop_duplicates()
    .sort_values("geometry_branch")
    .reset_index(drop=True)
)

geometry_weight_sum = float(
    geometry_validation["geometry_weight"].sum()
)

print("Cascadia geometry branches:")
display(geometry_validation)

print(
    "Geometry weight sum:",
    geometry_weight_sum,
)


if not abs(geometry_weight_sum - 1.0) < 1e-10:
    raise ValueError(
        "Cascadia geometry weights do not sum to 1."
    )

#validate oregon spatial pdf


oregon_pdf_validation = pd.DataFrame(
    [
        {
            "check": "Number of grid points",
            "expected": 821,
            "actual": int(
                oregon_pdf_summary.loc[
                    0,
                    "number_of_grid_points",
                ]
            ),
        },
        {
            "check": "Number of depth levels",
            "expected": 3,
            "actual": int(
                oregon_pdf_summary.loc[
                    0,
                    "number_of_depth_levels",
                ]
            ),
        },
        {
            "check": "Minimum depth, km",
            "expected": 42.0,
            "actual": float(
                oregon_pdf_summary.loc[
                    0,
                    "minimum_depth_km",
                ]
            ),
        },
        {
            "check": "Maximum depth, km",
            "expected": 60.0,
            "actual": float(
                oregon_pdf_summary.loc[
                    0,
                    "maximum_depth_km",
                ]
            ),
        },
    ]
)

oregon_pdf_validation["passes"] = (
    oregon_pdf_validation["expected"]
    == oregon_pdf_validation["actual"]
)

print("Oregon intraslab spatial PDF validation:")
display(oregon_pdf_validation)


pdf_sum = float(
    oregon_pdf_summary.loc[0, "pdf_sum"]
)

if not abs(pdf_sum - 1.0) < 1e-8:
    raise ValueError(
        f"Oregon spatial PDF sum is invalid: {pdf_sum}"
    )


print("=" * 72)
print("NOTEBOOK 1 VALIDATION COMPLETE")
print("=" * 72)

print(
    "\nThe USGS NSHM 2018 model was downloaded, "
    "verified, inventoried, and inspected successfully."
)

print(
    "\nVerified Cascadia rupture sets:",
    len(cascadia_rupture_sets),
)

print(
    "Verified Oregon intraslab grid points:",
    int(
        oregon_pdf_summary.loc[
            0,
            "number_of_grid_points",
        ]
    ),
)

print(
    "\nNext notebook:"
    "\n02_extract_usgs_rupture_rates.ipynb"
)

,description,file_name,exists,size_kb
0,Complete USGS file manifest,usgs_nshm_2018_file_manifest.csv,True,99.333008
1,Subduction file manifest,usgs_nshm_2018_subduction_manifest.csv,True,12.265625
2,Cascadia geometry logic tree,cascadia_geometry_logic_tree.csv,True,0.090820
3,Cascadia MFD branches,cascadia_mfd_branches.csv,True,2.591797
4,Cascadia partial-rupture logic tree,cascadia_partial_rupture_logic_tree.csv,True,0.677734
5,Oregon intraslab branches,oregon_intraslab_model_branches.csv,True,0.564453
6,Cascadia rupture-set inventory,cascadia_rupture_set_inventory.csv,True,4.854492
7,Oregon spatial PDF summary,oregon_intraslab_spatial_pdf_summary.csv,True,0.212891


Cascadia rupture-set count validation:


,rupture_category,expected_count,actual_count,passes
0,Full-margin,3,3,True
1,Segmented partial,12,12,True
2,Unsegmented partial,6,6,True


Cascadia geometry branches:


,geometry_branch,geometry_weight
0,bottom,0.3
1,middle,0.5
2,top,0.2


Geometry weight sum: 1.0
Oregon intraslab spatial PDF validation:


,check,expected,actual,passes
0,Number of grid points,821.0,821.0,True
1,Number of depth levels,3.0,3.0,True
2,"Minimum depth, km",42.0,42.0,True
3,"Maximum depth, km",60.0,60.0,True


NOTEBOOK 1 VALIDATION COMPLETE

The USGS NSHM 2018 model was downloaded, verified, inventoried, and inspected successfully.

Verified Cascadia rupture sets: 21
Verified Oregon intraslab grid points: 821

Next notebook:
02_extract_usgs_rupture_rates.ipynb
